# **Library Setup**

In [1]:
import torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using Device: {device}")
if device == 'cuda':
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

Using Device: cuda
GPU Name: NVIDIA GeForce RTX 4060 Laptop GPU


In [6]:
# CELL 1: Text to Raw UTF-8 Bytes
text = """Hello, Helia! Welcome to the DAI project.
We are building a Byte-Pair Encoding tokenizer from scratch to understand how Large Language Models compress text."""

# Encode the string to raw UTF-8 bytes, then convert to a list of integers (0-255)
tokens = list(text.encode("utf-8"))

print(f"Original text length: {len(text)} characters")
print(f"Tokenized length: {len(tokens)} bytes")
print("First 15 raw byte tokens:", tokens[:15])

Original text length: 156 characters
Tokenized length: 156 bytes
First 15 raw byte tokens: [72, 101, 108, 108, 111, 44, 32, 72, 101, 108, 105, 97, 33, 32, 87]


In [4]:
# CELL 2: The Stats Function
def get_stats(ids):
    counts = {}
    # Iterate through the list of IDs, looking at pairs (i, i+1)
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

stats = get_stats(tokens)

# Find the most frequent pair
top_pair = max(stats, key=stats.get)
print(f"The most frequent pair of byte IDs is {top_pair}, occurring {stats[top_pair]} times.")

The most frequent pair of byte IDs is (101, 32), occurring 6 times.


In [7]:
# CELL 3: The Merge Function
def merge(ids, pair, idx):
    # ids: the list of current tokens
    # pair: the specific tuple we want to replace (e.g., (101, 32))
    # idx: the new integer ID we will assign to this pair
    
    newids = []
    i = 0
    while i < len(ids):
        # If we are not at the very end AND the current pair matches our target
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2 # Skip the next token since we just merged it
        else:
            newids.append(ids[i])
            i += 1 # Move forward by 1
            
    return newids

# Let's test merging the top pair we found in Cell 2!
new_token_id = 256
tokens_merged = merge(tokens, top_pair, new_token_id)
print(f"Length before merge: {len(tokens)}")
print(f"Length after merge: {len(tokens_merged)}")
print(f"Compression achieved: {len(tokens) - len(tokens_merged)} tokens saved!")

Length before merge: 156
Length after merge: 150
Compression achieved: 6 tokens saved!


In [8]:
# CELL 4: Training the BPE Tokenizer
vocab_size = 276 # Target vocabulary size (256 base bytes + 20 new merges for our test)
num_merges = vocab_size - 256

# Reset tokens to the original UTF-8 bytes for the full training loop
ids = list(text.encode("utf-8")) 

merges = {} # This dictionary will store our "Merge Rules" to use later

for i in range(num_merges):
    stats = get_stats(ids)
    if not stats: 
        break # Safety break if there's nothing left to merge
        
    pair = max(stats, key=stats.get)
    idx = 256 + i # Create a new ID starting from 256, 257, 258...
    
    ids = merge(ids, pair, idx)
    merges[pair] = idx # Save the rule!
    
    print(f"Merge {i+1}: {pair} -> ID {idx} | New sequence length: {len(ids)}")

print("\nTraining complete! We now have a custom BPE vocabulary.")

Merge 1: (101, 32) -> ID 256 | New sequence length: 150
Merge 2: (101, 108) -> ID 257 | New sequence length: 146
Merge 3: (32, 116) -> ID 258 | New sequence length: 142
Merge 4: (99, 111) -> ID 259 | New sequence length: 139
Merge 5: (110, 103) -> ID 260 | New sequence length: 136
Merge 6: (72, 257) -> ID 261 | New sequence length: 134
Merge 7: (259, 109) -> ID 262 | New sequence length: 132
Merge 8: (112, 114) -> ID 263 | New sequence length: 130
Merge 9: (116, 46) -> ID 264 | New sequence length: 128
Merge 10: (97, 114) -> ID 265 | New sequence length: 126
Merge 11: (100, 105) -> ID 266 | New sequence length: 124
Merge 12: (266, 260) -> ID 267 | New sequence length: 122
Merge 13: (114, 32) -> ID 268 | New sequence length: 120
Merge 14: (258, 111) -> ID 269 | New sequence length: 118
Merge 15: (110, 100) -> ID 270 | New sequence length: 116
Merge 16: (103, 256) -> ID 271 | New sequence length: 114
Merge 17: (261, 108) -> ID 272 | New sequence length: 113
Merge 18: (272, 111) -> ID 273

In [12]:
# CELL 5: The Decoding Vocabulary
# 1. Start with the foundational 256 raw UTF-8 bytes
vocab = {idx: bytes([idx]) for idx in range(256)}

# 2. Add our custom merged tokens by combining the bytes of their pairs
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

print(f"Total Vocabulary Size: {len(vocab)}")
print(f"Bytes for our first custom token (ID 256): {vocab.get(256)}")

Total Vocabulary Size: 276
Bytes for our first custom token (ID 256): b'e '


In [13]:
# CELL 6: Encode and Decode
def decode(ids):
    # Retrieve the byte sequences for each ID and join them together
    raw_bytes = b"".join(vocab[idx] for idx in ids)
    # Convert bytes back to a string, ignoring broken characters
    return raw_bytes.decode("utf-8", errors="replace")

def encode(text):
    # Start with raw UTF-8 bytes
    tokens = list(text.encode("utf-8"))
    
    # Keep merging as long as we have at least 2 tokens
    while len(tokens) >= 2:
        # Get the current pairs in the sequence
        stats = get_stats(tokens)
        
        # Find the pair that we learned to merge earliest (lowest ID in our merges dict)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        
        # If the pair we found isn't in our merge rules, we are completely done!
        if pair not in merges:
            break
            
        # Apply the merge
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
        
    return tokens

# The Ultimate Test
test_text = "Hello, Helia! Welcome to the DAI project."
encoded_text = encode(test_text)
decoded_text = decode(encoded_text)

print(f"Original Text: {test_text}")
print(f"Encoded IDs:   {encoded_text}")
print(f"Decoded Text:  {decoded_text}")
print(f"Was the test successful? {test_text == decoded_text}")

Original Text: Hello, Helia! Welcome to the DAI project.
Encoded IDs:   [275, 261, 105, 97, 33, 32, 87, 257, 262, 256, 116, 111, 258, 104, 256, 68, 65, 73, 32, 263, 111, 106, 101, 99, 264]
Decoded Text:  Hello, Helia! Welcome to the DAI project.
Was the test successful? True
